# MobileNetV3-Small — تدريب لتحسين المتانة، مرحلة جديدة

شغّل خلية الكود الوحيدة بعد تفعيل GPU وإضافة صور QR وCSV الأصلي إلى Input. لا تعتمد الخلية على متغيرات الدفتر السابق.

تبدأ من أوزان الحقبة الرابعة السليمة:
`/kaggle/working/mobilenet_continue/full_train_phase1_fp32/best_epoch_04_10a6296afd7d.pt`
وتتطلب إيصال SHA256 المجاور لها (`.json`). احتفظ بالمجلد السابق كاملًا. لا تستبدل خلية تعمل حاليًا ولا تشغّل نسختين بالتوازي.

## الإعدادات

- نفس البنية ونفس تقسيم البيانات: Train710746، Validation148529. لا تُقيَّم صور Test.
- FP32 دون AMP. مُحسِّن AdamW جديد؛ هذه ليست استعادة مُحسِّن المرحلة السابقة.
- حتى5 حقب، وتوقف بعد حقبتين دون تحسن مؤهل.
- احتمال50% لتطبيق التعزيز على صورة التدريب؛ يختار حجم224 أو216 أو208، مع Nearest أوBilinear عند التصغير، ثم إزاحة حتى4 بكسلات مع خلفية بيضاء. قد يختار التحويل المحايد ضمن هذا الاحتمال. تُقيَّد الإزاحة للحفاظ على كل بكسلات المحتوى غير الأبيض. لا تدوير أو قلب أو قص عشوائي.
- قد يغيّر التصغير قابلية فك بعض الرموز. لم نفك مليون صورة أثناء التدريب؛ الحفاظ على المحتوى الدلالي لا تضمنه زيادة البيانات وحدها.

## التقييم والاختيار المحددان قبل التشغيل

تُقاس Validation الأصلية كاملة، ثم عينة ثابتة متوازنة1000 صورة تحت الحالات السبع في الاختبار التشخيصي: الأصل، إزاحات أربعة اتجاهات، وتصغيرNearest وBilinear.

درجة اختيار مخصصة = نصف Macro-F1 على كامل Validation الأصلي + نصف متوسط Macro-F1 للحالات الست المحوّلة. المتوسط عبر التحويلات ليس نتيجة على6000 صورة مستقلة؛ الصور نفسها متكررة.

لا تؤهَّل الأوزان الجديدة إذا انخفض Macro-F1 الأصلي أكثر من نقطة مئوية واحدة عن أوزان بداية هذه المرحلة. هذه سماحية عملية محددة مسبقًا، وليست ضمانًا أو معيارًا بحثيًا موحدًا. تُحفظ أوزان البداية كأفضل خيار مبدئي. كل حقبة مكتملة تُحفظ على أي حال، حتى إن لم تؤهَّل للاختيار. لا تُغير القاعدة بعد رؤية النتائج دون تسمية تجربة جديدة.

العينة التشخيصية معروفة وقد أثرت في تصميم التعزيزات. نتائجها تقيس التحسن على الحالات المحددة؛ لا تثبت المتانة أمام تحويلات لم نختبرها أو أمام مصادر خارجية. Test سبق استخدامه لتقرير خط الأساس؛ لا ندّعي أنه مجموعة جديدة لم تُفتح. أي تحسن مستقبلي يعرض مع سجل الاختيار والتحسين وبوضوح.

## المتابعة والحفظ

تظهر مراحل التحضير، اختبار الحفظ، قياسات البداية، ثم شريط التدريب. التقييم الأول قبل التدريب قد يستغرق عدة دقائق. عند توقف التنفيذ، أعد تشغيل الخلية نفسها؛ تعاد الحقبة غير المكتملة من آخر حقبة محفوظة سليمة.

مجلد هذه التجربة: `/kaggle/working/mobilenet_continue/robust_aug_phase2_v1`.
يحتوي أوزانًا مستقلة لكل حفظ، إيصالات البصمة، history.json، عينة Validation، معاينة التعزيزات، وجدول validation_comparison.csv عند اكتمال الخلية. نزّل نسخًا خارج الجلسة بعد الحفظ؛ الفحص الفوري لا يضمن دوام الملفات لاحقًا.

تم فحص صياغة الدفتر ودوال التحويل محليًا. لم يُنفذ تدريب GPU أو تحميل ملفات Kaggle في بيئة المساعد. لا ضمان لزيادة الدقة أو المتانة؛ سيقرر التقييم ذلك.


In [ ]:
print('MobileNet robustness training started — loading libraries...', flush=True)
import os, json, time, gc, math, random, uuid, shutil, hashlib, sqlite3, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image, ImageChops
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

# All settings are here. Re-running this cell resumes only this NEW phase.
SOURCE = Path('/kaggle/working/mobilenet_continue/full_train_phase1_fp32/best_epoch_04_10a6296afd7d.pt')
ROOT = Path('/kaggle/working/qr_visual_fresh')
RUN = Path('/kaggle/working/mobilenet_continue/robust_aug_phase2_v1')
RESUME = True
CFG = dict(seed=42, epochs=5, patience=2, batch_size=32, eval_batch_size=128,
           backbone_lr=1e-5, head_lr=2e-5, weight_decay=1e-5,
           label_smoothing=0.1, img_size=224, clip=1.0, num_workers=0,
           amp=False, threshold=0.5, max_train=None,
           augment_probability=0.5, train_sizes=[208,216,224], max_shift=4,
           robust_per_class=500, clean_tolerance=0.01,
           selection='0.5*full_clean_macro_f1 + 0.5*mean_six_transformed_macro_f1',
           augmentation_version=1)
# num_workers=0 is deliberate: first progress is visible without worker startup.
# There is no Test evaluation, threshold search, random split, or ImageNet download.
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Settings, then rerun this cell.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
RUN.mkdir(parents=True, exist_ok=True)
assert SOURCE.is_file(), f'Missing source checkpoint: {SOURCE}'
def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def atomic_json(value,path):
    path=Path(path);tmp=path.with_name(path.name+'.writing')
    with tmp.open('w') as f:
        json.dump(value,f,indent=2,allow_nan=False)
        f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)

def atomic_parquet(frame,path):
    path=Path(path);tmp=path.with_name(path.name+'.writing')
    frame.to_parquet(tmp,index=False)
    import pyarrow.parquet as pq
    assert pq.ParquetFile(tmp).metadata.num_rows==len(frame)
    os.replace(tmp,path)

# Recover the manifest only; training/model files are never modified.
import sqlite3
from concurrent.futures import ThreadPoolExecutor

def training_signature(frame):
    h=hashlib.sha256()
    for row in frame[['path','label','split','url','ok','sha256']].itertuples(index=False,name=None):
        h.update((json.dumps(row,ensure_ascii=False,separators=(',',':'))+'\n').encode())
    return h.hexdigest()

def clean_from_hashes(raw,hashes,audit):
    assert hashes.path.is_unique
    joined=raw.merge(hashes[['path','sha256']],on='path',how='left',validate='one_to_one')
    assert joined.sha256.notna().all(),'Incomplete image hash coverage.'
    assert joined.sha256.str.fullmatch('[0-9a-f]{64}').all()
    grouped=joined.groupby('sha256').label.nunique()
    conflicts=set(grouped.index[grouped.gt(1)])
    cleaned=joined[~joined.sha256.isin(conflicts)].copy()
    cleaned['_priority']=cleaned.split.map({'train':0,'val':1,'test':2})
    assert cleaned['_priority'].notna().all()
    cleaned=cleaned.sort_values(['_priority','path']).drop_duplicates('sha256').drop(columns='_priority')
    cleaned=cleaned.sort_values('path').reset_index(drop=True)
    assert len(cleaned)==audit['retained_samples'],'Restored membership differs.'
    assert training_signature(cleaned)==audit['data_signature'], \
        'Restored data does not match training; do not remove this assertion.'
    return cleaned

def recover_manifest(root,audit):
    assert audit.get('clean_duplicates') is True
    folder=root/'verified_mobilenet_inputs';folder.mkdir(parents=True,exist_ok=True)
    # Prefer a newly verified copy if a previous repair finished.
    for path,kind in [(folder/'splits_verified.parquet','parquet'),(folder/'splits_verified.csv.gz','csv')]:
        if not path.is_file():continue
        try:
            frame=pd.read_parquet(path) if kind=='parquet' else pd.read_csv(path,keep_default_na=False)
            assert len(frame)==audit['retained_samples'] and frame.path.is_unique
            assert training_signature(frame)==audit['data_signature']
            print('Loaded recovered manifest:',path,flush=True)
            return frame,path
        except Exception as exc:
            print('Cannot reuse recovered copy:',path.name,type(exc).__name__,flush=True)
    candidates=[Path('/kaggle/input/datasets/feras2004/spliteoriginal/splitsOriginal.csv'),
                Path('/kaggle/input/datasets/feras2004/splits/splits.csv')]
    original_csv=None
    for path in candidates:
        if path.is_file() and file_hash(path)==audit.get('source_splits_sha256'):
            original_csv=path;break
    assert original_csv is not None, 'Original Input CSV missing or different; restore the original source first.'
    print('Recovery: loading verified original CSV...',flush=True)
    raw=pd.read_csv(original_csv,keep_default_na=False,
                    dtype={'path':str,'url':str,'domain':str,'split':str})
    assert len(raw)==audit['original_samples'] and raw.path.is_unique
    for col,values in [('label',[0,1]),('ok',[0,1]),('fold',list(range(20)))]:
        parsed=pd.to_numeric(raw[col],errors='coerce')
        assert parsed.isin(values).all(),f'Invalid source {col}.'
        raw[col]=parsed.astype('int64')
    assert set(raw.split)=={'train','val','test'}
    expected=np.where(raw.fold.lt(3),'test',np.where(raw.fold.lt(6),'val','train'))
    assert np.array_equal(expected,raw.split.to_numpy())
    roots=audit.get('image_roots')
    assert isinstance(roots,list) and len(roots)==2
    raw['source_path']=raw.path
    def rebound(row):
        marker='QR_All_benign/qrs/' if row.label==0 else 'QR_All_Malicious/qrs/'
        assert marker in row.path,'Source path/class mismatch.'
        rel=row.path.split(marker,1)[1]
        assert not Path(rel).is_absolute() and '..' not in Path(rel).parts
        return str(Path(roots[int(row.label)])/rel)
    raw['path']=[rebound(row) for row in raw.itertuples(index=False)]
    assert raw.path.is_unique
    restored=None
    old_cache=root/'hashes.sqlite'
    if old_cache.is_file():
        print('Recovery: trying saved hash database (read-only)...',flush=True)
        try:
            with sqlite3.connect(old_cache.resolve().as_uri()+'?mode=ro',uri=True) as con:
                assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
                hashes=pd.read_sql_query("SELECT path,sha256 FROM hashes WHERE error=''",con)
            restored=clean_from_hashes(raw,hashes,audit)
            print('Recovered using existing hashes; images were not re-read.',flush=True)
        except Exception as exc:
            print('Saved hashes unusable:',type(exc).__name__,str(exc)[:200],flush=True)
    if restored is None:
        print('Recovery: rehashing image bytes; no QR decoding or training. This may take time.',flush=True)
        # New database only; keep the old damaged one unchanged.
        new_cache=folder/'hashes_rebuilt.sqlite'
        con=sqlite3.connect(new_cache)
        try:
            assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
        except Exception:
            con.close()
            new_cache=folder/f'hashes_rebuilt_{time.time_ns()}.sqlite'
            con=sqlite3.connect(new_cache)
        con.execute('PRAGMA synchronous=FULL')
        con.execute('CREATE TABLE IF NOT EXISTS hashes (path TEXT PRIMARY KEY,sha256 TEXT)')
        con.commit()
        known={p:h for p,h in con.execute('SELECT path,sha256 FROM hashes')}
        pending=[p for p in raw.path if p not in known]
        def read_hash(path):return path,file_hash(path)
        with ThreadPoolExecutor(max_workers=4) as pool:
            bar=tqdm(total=len(raw),initial=len(raw)-len(pending),desc='Recover image hashes',mininterval=1)
            try:
                for start in range(0,len(pending),2000):
                    batch=list(pool.map(read_hash,pending[start:start+2000]))
                    con.executemany('INSERT OR REPLACE INTO hashes VALUES (?,?)',batch)
                    con.commit();bar.update(len(batch))
            finally:bar.close()
        hashes=pd.read_sql_query('SELECT path,sha256 FROM hashes',con);con.close()
        restored=clean_from_hashes(raw,hashes,audit)
    print('Recovered content matches the original training signature.',flush=True)
    parquet=folder/'splits_verified.parquet'
    atomic_parquet(restored,parquet)
    checked=pd.read_parquet(parquet)
    assert training_signature(checked)==audit['data_signature']
    # A different storage format supplies an additional recovery copy.
    csv=folder/'splits_verified.csv.gz';tmp=folder/'splits_verified.csv.gz.writing'
    restored.to_csv(tmp,index=False,compression='gzip');os.replace(tmp,csv)
    atomic_json(dict(data_signature=audit['data_signature'],source_csv=str(original_csv),
                     source_sha256=file_hash(original_csv),parquet_sha256=file_hash(parquet),
                     retained_samples=len(restored)),folder/'recovery_record.json')
    return restored,parquet


print('Checking selected phase-1 checkpoint and its receipt...', flush=True)
source_receipt=json.loads(SOURCE.with_suffix('.json').read_text())
assert source_receipt['file']==SOURCE.name
SOURCE_SHA=source_receipt['sha256']
assert file_hash(SOURCE)==SOURCE_SHA, 'Source checkpoint changed; do not bypass.'
original = torch.load(SOURCE, map_location='cpu', weights_only=False)  # your own trusted checkpoint only
assert original['epoch']==4
assert original['cfg']['max_train'] is None
assert original['data_signature']=='d4509181f834a16144ad45e24b8ce686a4e608e07a9846a880757c705739d1ff'
assert original['cfg']['img_size'] == CFG['img_size']
backup = RUN / 'source_original.pt'
if not backup.exists():
    temporary = RUN / ('source_' + uuid.uuid4().hex + '.writing')
    shutil.copy2(SOURCE, temporary)
    assert file_hash(temporary) == SOURCE_SHA
    os.replace(temporary, backup)
assert file_hash(backup) == SOURCE_SHA
assert file_hash(SOURCE) == SOURCE_SHA

print('Loading saved data manifest (preparation, not training yet)...', flush=True)
audit = json.loads((ROOT/'audit.json').read_text())
assert audit.get('train_ready') is True and audit.get('clean_duplicates') is True
EXPECTED_SIGNATURE = 'd4509181f834a16144ad45e24b8ce686a4e608e07a9846a880757c705739d1ff'
assert audit['data_signature'] == EXPECTED_SIGNATURE, 'Unexpected audited training membership.'
manifest = ROOT/'verified_mobilenet_inputs/splits_verified.parquet'
try:
    df = pd.read_parquet(manifest)
    assert training_signature(df) == EXPECTED_SIGNATURE
except Exception as exc:
    print('Prepared manifest unavailable:', type(exc).__name__, str(exc)[:180], flush=True)
    df, manifest = recover_manifest(ROOT, audit)
assert len(df) == 1005700 and df.path.is_unique
assert df.label.isin([0,1]).all() and df.ok.isin([0,1]).all()
assert training_signature(df) == EXPECTED_SIGNATURE
# Verify all original metadata; do not trust repaired labels or new membership.
original_csv = Path('/kaggle/input/datasets/feras2004/spliteoriginal/splitsOriginal.csv')
assert original_csv.is_file(), f'Add immutable source CSV: {original_csv}'
assert file_hash(original_csv) == audit['source_splits_sha256'], 'Source CSV changed.'
fields = ['url','ok','label','domain','fold','split']
raw = pd.read_csv(original_csv, keep_default_na=False, usecols=['path']+fields)
assert raw.path.is_unique and len(raw) == audit['original_samples']
source_keys = df['source_path'] if 'source_path' in df else df['path']
assert source_keys.is_unique
aligned = raw.set_index('path').loc[source_keys]
for col in fields:
    if col in ['ok','label','fold']:
        assert np.array_equal(pd.to_numeric(df[col]).to_numpy(), pd.to_numeric(aligned[col]).to_numpy()), col
    else:
        assert np.array_equal(df[col].fillna('').astype(str).to_numpy(), aligned[col].fillna('').astype(str).to_numpy()), col
assert df.groupby('sha256').split.nunique().max() == 1
assert df.groupby('sha256').label.nunique().max() == 1
assert df.groupby('domain', dropna=False).split.nunique().max() == 1
assert df[df.ok.eq(1)].groupby('url').split.nunique().max() == 1
train = df[df.split.eq('train')].reset_index(drop=True)
val = df[df.split.eq('val')].reset_index(drop=True)
assert train.label.value_counts().to_dict() == {1:403324, 0:307422}
assert val.label.value_counts().to_dict() == {1:86299, 0:62230}
assert len(train)==710746 and len(val)==148529
print(f'Data verified: Train={len(train):,}, Validation={len(val):,}. Test images will not be loaded.', flush=True)
del raw, aligned, df
gc.collect()

class MobileNetVisual(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual_backbone = torchvision.models.mobilenet_v3_small(weights=None)
        self.visual_backbone.classifier = nn.Identity()
        self.visual_proj = nn.Sequential(nn.Linear(576,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3))
        self.fusion = nn.Sequential(nn.Linear(256,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
                                    nn.Linear(256,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128,2))
    def forward(self, x):
        return self.fusion(self.visual_proj(self.visual_backbone(x)))

random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
model = MobileNetVisual()
model.load_state_dict(original['model_state'], strict=True)
for key, value in model.state_dict().items():
    if value.is_floating_point():
        assert torch.isfinite(value).all(), f'Nonfinite original weight: {key}'
print('PASS: every original model tensor loaded with strict=True; all tensors are finite.', flush=True)
print('Precision: FP32 (AMP disabled).', flush=True)
model.to(DEVICE)
del original

# No rotation, crop augmentation, flips, color jitter or blur.
# Translation only removes white border pixels; all nonwhite pixels are retained.
RESIZE=T.Resize((224,224))
NORMALIZE=T.Compose([T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])

def shrink_center(image,size,method):
    small=image.resize((size,size),resample=method)
    canvas=Image.new('RGB',(224,224),'white')
    canvas.paste(small,((224-size)//2,(224-size)//2))
    return canvas

def safe_random_shift(image,rng):
    bbox=ImageChops.difference(image,Image.new('RGB',image.size,'white')).getbbox()
    if bbox is None:return image.copy()
    left,top,right,bottom=bbox
    limit=CFG['max_shift']
    dx=rng.randint(max(-limit,-left),min(limit,224-right))
    dy=rng.randint(max(-limit,-top),min(limit,224-bottom))
    canvas=Image.new('RGB',(224,224),'white')
    canvas.paste(image,(dx,dy))
    return canvas

def augment_image(image,rng=random):
    if rng.random()>=CFG['augment_probability']:return image.copy()
    size=rng.choice(CFG['train_sizes'])
    if size!=224:
        method=rng.choice([Image.Resampling.NEAREST,Image.Resampling.BILINEAR])
        image=shrink_center(image,size,method)
    return safe_random_shift(image,rng)

ROBUST_VARIANTS=['original','translate_left_4','translate_right_4',
                 'translate_up_4','translate_down_4','shrink_nearest_208','shrink_bilinear_208']

def validation_image(image,variant):
    if variant=='original':return image
    shifts={'translate_left_4':(-4,0),'translate_right_4':(4,0),
            'translate_up_4':(0,-4),'translate_down_4':(0,4)}
    if variant in shifts:
        result=Image.new('RGB',(224,224),'white');result.paste(image,shifts[variant])
        before=np.any(np.asarray(image)!=255,axis=2).sum()
        after=np.any(np.asarray(result)!=255,axis=2).sum()
        assert before==after,'Robustness transform would crop nonwhite pixels.'
        return result
    assert variant in ['shrink_nearest_208','shrink_bilinear_208']
    method=Image.Resampling.NEAREST if variant=='shrink_nearest_208' else Image.Resampling.BILINEAR
    return shrink_center(image,208,method)

class QRDataset(Dataset):
    def __init__(self,frame,augment=False,variant='original'):
        self.paths=frame.path.to_numpy();self.labels=frame.label.to_numpy(dtype=np.int64)
        self.augment=augment;self.variant=variant
    def __len__(self):return len(self.paths)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as image:
            image=RESIZE(image.convert('RGB'))
        image=augment_image(image) if self.augment else validation_image(image,self.variant)
        return NORMALIZE(image),int(self.labels[i])

# Same deterministic balanced Validation sample as the diagnostic experiment.
robust_sample=pd.concat([val[val.label.eq(label)].sample(n=CFG['robust_per_class'],random_state=42)
                         for label in [0,1]]).sort_values('path').reset_index(drop=True)
robust_sample.to_csv(RUN/'robust_validation_membership.csv',index=False)
# Visual previews use a private RNG; they do not consume training RNG state.
preview_rng=random.Random(2026)
preview=Image.new('RGB',(224*5,224*4),'white')
for i,row in enumerate(train.sample(n=4,random_state=42).itertuples(index=False)):
    with Image.open(row.path) as image:base=RESIZE(image.convert('RGB'))
    preview.paste(base,(0,224*i))
    for j in range(1,5):preview.paste(augment_image(base,preview_rng),(224*j,224*i))
preview.save(RUN/'training_augmentation_preview.png')

generator = torch.Generator().manual_seed(CFG['seed'])
train_loader = DataLoader(QRDataset(train,augment=True), batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, generator=generator, drop_last=False)
val_loader = DataLoader(QRDataset(val), batch_size=CFG['eval_batch_size'], shuffle=False,
                        num_workers=CFG['num_workers'], pin_memory=True)
assert len(train) % CFG['batch_size'] != 1, 'BatchNorm cannot train on a final batch of one.'
optimizer = torch.optim.AdamW([
    {'params':model.visual_backbone.parameters(), 'lr':CFG['backbone_lr']},
    {'params':list(model.visual_proj.parameters())+list(model.fusion.parameters()), 'lr':CFG['head_lr']}
], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=CFG['amp'])
criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])

# Immutable checkpoints: each successful save has its own filename and receipt.
def state_cpu():
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def verified_save(payload, label):
    path = RUN / (label + '_' + uuid.uuid4().hex[:12] + '.pt')
    temporary = path.with_suffix('.writing')
    print('Saving and rereading:', path.name, flush=True)
    with temporary.open('xb') as f:
        torch.save(payload, f); f.flush(); os.fsync(f.fileno())
    with zipfile.ZipFile(temporary) as archive:
        assert archive.testzip() is None, 'Checkpoint ZIP CRC failed.'
    restored = torch.load(temporary, map_location='cpu', weights_only=False)
    target = payload['model_state']
    assert set(restored['model_state']) == set(target)
    for key,value in target.items():
        assert torch.equal(restored['model_state'][key], value.cpu()), f'Saved tensor mismatch: {key}'
    # Restore a separate model strictly, then compare its actual output.
    verification_rng = torch.get_rng_state()
    probe = MobileNetVisual().eval()
    probe.load_state_dict(restored['model_state'], strict=True)
    reference = MobileNetVisual().eval()
    reference.load_state_dict(target, strict=True)
    with torch.no_grad():
        probe_x = torch.zeros(1,3,224,224)
        assert torch.equal(probe(probe_x), reference(probe_x))
    if 'optimizer' in payload:
        assert restored['optimizer']['param_groups'] == payload['optimizer']['param_groups']
        assert set(restored['optimizer']['state']) == set(payload['optimizer']['state'])
        for index, entry in payload['optimizer']['state'].items():
            for key, value in entry.items():
                saved_value = restored['optimizer']['state'][index][key]
                if torch.is_tensor(value): assert torch.equal(saved_value, value.cpu())
                else: assert saved_value == value
    del restored, probe, reference
    torch.set_rng_state(verification_rng)
    checksum = file_hash(temporary)
    os.replace(temporary, path)
    assert file_hash(path) == checksum
    atomic_json(dict(file=path.name, sha256=checksum, label=label,
                     epoch=payload.get('epoch'), completed=time.time()), path.with_suffix('.json'))
    print('PASS: saved checkpoint verified:', path.name, flush=True)
    return path

def full_checkpoint(epoch, best, best_file, bad_epochs, history):
    return dict(format_version=1, mode='visual', model_state=state_cpu(), cfg=CFG,
        data_signature=EXPECTED_SIGNATURE, source_sha256=SOURCE_SHA,
        epoch=epoch, optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
        scaler=scaler.state_dict(), best_selection_score=best, best_file=best_file,
        bad_epochs=bad_epochs, history=history,
        python_rng=random.getstate(), numpy_rng=np.random.get_state(),
        torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all(),
        loader_rng=generator.get_state())

CLEAN_REFERENCE=None

def evaluate_loader(loader,label):
    model.eval();ys=[];scores=[]
    with torch.no_grad():
        for x,y in tqdm(loader,desc=label,mininterval=2):
            logits=model(x.to(DEVICE,non_blocking=True))
            scores.append(logits.float().softmax(1)[:,1].cpu().numpy());ys.append(y.numpy())
    y=np.concatenate(ys);p=np.concatenate(scores)
    assert np.isfinite(p).all()
    pred=(p>=CFG['threshold']).astype(np.int64)
    return dict(accuracy=float(accuracy_score(y,pred)),macro_f1=float(f1_score(y,pred,average='macro')),
        malicious_f1=float(f1_score(y,pred,zero_division=0)),precision=float(precision_score(y,pred,zero_division=0)),
        recall=float(recall_score(y,pred,zero_division=0)),auc=float(roc_auc_score(y,p)),
        confusion_matrix=confusion_matrix(y,pred,labels=[0,1]).tolist())

def evaluate(label):
    metrics=evaluate_loader(val_loader,label+' clean/full')
    transformed={}
    for variant in ROBUST_VARIANTS:
        loader=DataLoader(QRDataset(robust_sample,variant=variant),batch_size=CFG['eval_batch_size'],
                          shuffle=False,num_workers=0,pin_memory=True)
        score=evaluate_loader(loader,label+' '+variant)
        transformed[variant]=score
        print(f"  {variant}: Acc={score['accuracy']:.4%} | Macro-F1={score['macro_f1']:.4%}",flush=True)
    mean_robust=float(np.mean([transformed[v]['macro_f1'] for v in ROBUST_VARIANTS if v!='original']))
    reference=metrics['macro_f1'] if CLEAN_REFERENCE is None else CLEAN_REFERENCE
    metrics.update(robust_validation=transformed,robust_macro_f1_mean=mean_robust,
                   selection_score=0.5*(metrics['macro_f1']+mean_robust),
                   clean_eligible=bool(metrics['macro_f1']>=reference-CFG['clean_tolerance']))
    print(f"{label}: Clean Acc={metrics['accuracy']:.4%} | Clean Macro-F1={metrics['macro_f1']:.4%} | "
          f"Robust mean Macro-F1={mean_robust:.4%} | Selection={metrics['selection_score']:.4%} | "
          f"Clean eligible={metrics['clean_eligible']}",flush=True)
    return metrics

identity=dict(cfg=CFG, data_signature=EXPECTED_SIGNATURE, source_sha256=SOURCE_SHA,
              versions=dict(torch=torch.__version__, torchvision=torchvision.__version__),
              phase='warm start from phase1 epoch4; fresh optimizer; robustness augmentation; Validation only')
config_path=RUN/'config.json'
if config_path.exists():
    saved=json.loads(config_path.read_text())
    assert saved == identity, 'Settings/version changed. Use a different RUN directory for a new experiment.'
else: atomic_json(identity, config_path)

# An interrupted epoch is repeated from the last VERIFIED completed epoch.
start_epoch=0; best=-1.; best_file=None; bad_epochs=0; history=[]; resumed=False
receipts=[]
for receipt in RUN.glob('epoch_*.json'):
    record=json.loads(receipt.read_text()); receipts.append((int(record['epoch']), receipt, record))
if receipts:
    assert RESUME, 'Run exists: set RESUME=True or choose a new RUN.'
    for _, receipt, record in sorted(receipts, key=lambda r:r[0], reverse=True):
        try:
            path=RUN/record['file']
            assert file_hash(path)==record['sha256'], 'Checksum mismatch'
            checkpoint=torch.load(path, map_location='cpu', weights_only=False)
            assert checkpoint['cfg']==CFG and checkpoint['source_sha256']==SOURCE_SHA
            assert checkpoint['data_signature']==EXPECTED_SIGNATURE
            # Make sure the selected best weights still exist and match their receipt.
            best_path=RUN/checkpoint['best_file']
            best_record=json.loads(best_path.with_suffix('.json').read_text())
            assert file_hash(best_path)==best_record['sha256']
        except Exception as exc:
            print('Cannot resume:', record['file'], type(exc).__name__, str(exc)[:150], flush=True)
            continue
        model.load_state_dict(checkpoint['model_state'], strict=True)
        optimizer.load_state_dict(checkpoint['optimizer'])
        scheduler.load_state_dict(checkpoint['scheduler']); scaler.load_state_dict(checkpoint['scaler'])
        random.setstate(checkpoint['python_rng']); np.random.set_state(checkpoint['numpy_rng'])
        torch.set_rng_state(checkpoint['torch_rng']); torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
        generator.set_state(checkpoint['loader_rng'])
        start_epoch=checkpoint['epoch']; best=checkpoint['best_selection_score']; best_file=checkpoint['best_file']
        bad_epochs=checkpoint['bad_epochs']; history=checkpoint['history']; resumed=True
        CLEAN_REFERENCE=history[0]['validation']['macro_f1']
        del checkpoint
        print(f'Resumed this phase after epoch {start_epoch}. An incomplete next epoch will repeat.', flush=True)
        break
    assert resumed, 'No verified continuation checkpoint. Original source remains untouched. Inspect files before restarting.'

if not resumed:
    # One real forward/backward/optimizer step, round-trip save, then restore all
    # initial tensors, RNG and optimizer state before the actual experiment.
    print('Storage + optimizer smoke test (not a training epoch)...', flush=True)
    initial=state_cpu()
    py_rng=random.getstate(); np_rng=np.random.get_state(); cpu_rng=torch.get_rng_state()
    gpu_rng=torch.cuda.get_rng_state_all(); loader_rng=generator.get_state()
    x,y=next(iter(train_loader)); model.train(); optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda',enabled=CFG['amp']):
        loss=criterion(model(x.to(DEVICE)),y.to(DEVICE))
    assert torch.isfinite(loss)
    scaler.scale(loss).backward(); scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), CFG['clip'], error_if_nonfinite=True)
    scaler.step(optimizer); scaler.update()
    verified_save(full_checkpoint(-1,-1,None,0,[]),'storage_smoke')
    model.load_state_dict(initial,strict=True); optimizer.state.clear()
    optimizer.zero_grad(set_to_none=True)
    scaler=torch.amp.GradScaler('cuda',enabled=CFG['amp'])
    random.setstate(py_rng); np.random.set_state(np_rng); torch.set_rng_state(cpu_rng)
    torch.cuda.set_rng_state_all(gpu_rng); generator.set_state(loader_rng)
    del initial,x,y,loss
    baseline=evaluate('Phase1 weights — BASELINE')
    CLEAN_REFERENCE=baseline['macro_f1']
    best=baseline['selection_score']
    history=[dict(epoch=0,stage='original_weights_baseline',validation=baseline)]
    best_file=verified_save(dict(model_state=state_cpu(),cfg=CFG,epoch=0,
        data_signature=EXPECTED_SIGNATURE,source_sha256=SOURCE_SHA,validation=baseline),'best_epoch_00').name
    verified_save(full_checkpoint(0,best,best_file,0,history),'epoch_00')
    atomic_json(history,RUN/'history.json')

print('TRAINING PHASE: batch progress below is actual training.',flush=True)
for epoch in range(start_epoch+1,CFG['epochs']+1):
    if bad_epochs>=CFG['patience']:
        print('Early stopping was already reached.',flush=True); break
    model.train(); seen=0; correct=0; total_loss=0.; started=time.time()
    lr_used=[g['lr'] for g in optimizer.param_groups]
    bar=tqdm(train_loader,total=len(train_loader),desc=f'MobileNet Train {epoch}/{CFG["epochs"]}',mininterval=2)
    for batch,(x,y) in enumerate(bar,1):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda',enabled=CFG['amp']):
            logits=model(x); loss=criterion(logits,y)
        assert torch.isfinite(loss), 'Nonfinite loss; stop without replacing any weights.'
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(),CFG['clip'],error_if_nonfinite=True)
        scaler.step(optimizer); scaler.update()
        n=y.size(0); seen+=n; total_loss+=float(loss.detach())*n
        correct+=int((logits.detach().argmax(1)==y).sum())
        if batch==1 or batch%50==0 or batch==len(train_loader):
            bar.set_postfix(loss=f'{total_loss/seen:.4f}',acc=f'{correct/seen:.2%}',images=f'{seen:,}/{len(train):,}')
        if batch==1 or batch%500==0 or batch==len(train_loader):
            speed=seen/max(time.time()-started,1e-6)
            print(f'[MobileNet] Epoch {epoch}/{CFG["epochs"]} | Batch {batch}/{len(train_loader)} | '
                  f'Images {seen:,}/{len(train):,} | {seen/len(train):.1%} | '
                  f'Loss {total_loss/seen:.4f} | Train Acc {correct/seen:.2%} | '
                  f'{speed:.1f} img/s | ETA {(len(train)-seen)/max(speed,1e-6)/60:.1f} min',flush=True)
    assert seen==len(train)
    metrics=evaluate(f'MobileNet Epoch {epoch} — VAL')
    improved=metrics['clean_eligible'] and metrics['selection_score']>best+1e-6
    if improved:
        best=metrics['selection_score']; bad_epochs=0
        best_file=verified_save(dict(model_state=state_cpu(),cfg=CFG,epoch=epoch,
            data_signature=EXPECTED_SIGNATURE,source_sha256=SOURCE_SHA,validation=metrics),f'best_epoch_{epoch:02d}').name
    else: bad_epochs+=1
    scheduler.step()
    history.append(dict(epoch=epoch,train_loss=total_loss/seen,train_accuracy=correct/seen,
                        learning_rates=lr_used,validation=metrics,improved=improved))
    verified_save(full_checkpoint(epoch,best,best_file,bad_epochs,history),f'epoch_{epoch:02d}')
    atomic_json(history,RUN/'history.json')
    print(f'VERIFIED epoch {epoch} saved. Best eligible selection score={best:.4%}. '
          f'Download checkpoints outside this session. No-improvement: {bad_epochs}/{CFG["patience"]}',flush=True)
    if bad_epochs>=CFG['patience']:
        print('Early stopping.',flush=True); break

assert file_hash(SOURCE)==SOURCE_SHA, 'Original source file has changed externally.'
print(f'Finished. Best eligible selection score={best:.4%}',flush=True)
print('Selected weights:',RUN/best_file,flush=True)
print('No Test images were evaluated in this phase. Phase1 weights were not overwritten.',flush=True)
print('Selection is a custom Validation objective, not Accuracy or Test performance.',flush=True)
comparison=[]
for row in history:
    m=row['validation']
    comparison.append(dict(epoch=row['epoch'],clean_accuracy=m['accuracy'],clean_macro_f1=m['macro_f1'],
        robust_macro_f1_mean=m['robust_macro_f1_mean'],selection_score=m['selection_score'],
        clean_eligible=m['clean_eligible'],improved=row.get('improved',False)))
pd.DataFrame(comparison).to_csv(RUN/'validation_comparison.csv',index=False)
print(pd.DataFrame(comparison).to_string(index=False),flush=True)
